# 🛡️ Kaggle 24/7 OSINT Agent + Tunnel Cloudflare & Keep-Alive
Plateforme OSINT 24/7 alimentée par Qwen GGUF. Boucle infinie active.

In [ ]:
# 1. Redirection TMPDIR et dossiers
import os
os.environ['TMPDIR'] = '/kaggle/working/tmp'
os.environ['PIP_CACHE_DIR'] = '/kaggle/working/tmp/pip'
os.makedirs('/kaggle/working/tmp', exist_ok=True)
os.makedirs('/kaggle/working/models', exist_ok=True)
print('🟢 Dossiers temporaires et modèles prêts !')

In [ ]:
# 2. Clonage ou maj Git
import os, subprocess
repo_dir = '/kaggle/working/projet_osint'
clone_url = 'https://github.com/whbky6vqjb-coder/osint.git'
if os.path.exists(repo_dir):
    print('⚡ Git pull...')
    subprocess.run(['git', '-C', repo_dir, 'pull', 'origin', 'main'], check=False)
else:
    print('📥 Git clone...')
    subprocess.run(['git', 'clone', clone_url, repo_dir], check=False)
!pip install --no-cache-dir --prefer-binary huggingface_hub "llama-cpp-python[server]"

In [ ]:
# 3. Lancement de llama_cpp.server et Cloudflare
import os, urllib.request, subprocess, time
from huggingface_hub import hf_hub_download

model_dir = '/kaggle/working/models'
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, 'qwen2.5-coder-7b-instruct-q4_k_m.gguf')

if not os.path.exists(model_path):
    print('📥 Téléchargement modèle GGUF...')
    try:
        model_path = hf_hub_download(repo_id='Qwen/Qwen2.5-Coder-7B-Instruct-GGUF', filename='qwen2.5-coder-7b-instruct-q4_k_m.gguf', local_dir=model_dir)
    except Exception as e:
        print('Download info:', e)

print('🚀 Lancement de llama_cpp.server (Mode CPU)...')
with open('/tmp/llama_server.log', 'w') as log_file:
    subprocess.Popen(['python3', '-m', 'llama_cpp.server', '--model', model_path, '--host', '127.0.0.1', '--port', '8080', '--n_gpu_layers', '0', '--n_ctx', '4096'], stdout=log_file, stderr=subprocess.STDOUT)
time.sleep(8)

cloudflared_bin = '/tmp/cloudflared'
if not os.path.exists(cloudflared_bin):
    print('Downloading cloudflared...')
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared_bin)
    os.chmod(cloudflared_bin, 0o755)

print('🌐 Démarrage du tunnel Cloudflare...')
with open('/tmp/cloudflared.log', 'w') as log_file:
    subprocess.Popen([cloudflared_bin, 'tunnel', '--url', 'http://127.0.0.1:8080'], stdout=log_file, stderr=subprocess.STDOUT)
time.sleep(8)

repo_dir = '/kaggle/working/projet_osint'
script_path = os.path.join(repo_dir, 'backend/app/cloud_sync/publish_llm_url.py')
if os.path.exists(script_path):
    print('Running publisher script...')
    subprocess.run(['python3', script_path], cwd=repo_dir)


In [ ]:
# 4. Boucle 24/7 avec Watchdog 11h
import time, subprocess, os
print('🟢 Serveur actif 24/7. Boucle d d\'écoute et Watchdog 11h activés !')
start_time = time.time()
eleven_hours = 11 * 3600
counter = 0
while True:
    time.sleep(60)
    counter += 1
    elapsed = time.time() - start_time
    if elapsed >= eleven_hours:
        print('⏰ 11h d uptime atteint. Signalement maintenance...')
        break
    if counter % 30 == 0:
        rem = round((eleven_hours - elapsed) / 3600, 1)
        print(f'[{time.strftime("%Y-%m-%d %H:%M:%S")} 🟢] Uptime: {counter}m ({rem}h restantes)')
